In [50]:
import numpy as np
import random
from collections import deque

# Define the memory buffer to store experience tuples
class ReplayBuffer():
    def __init__(self, buffer_size):
        self.buffer = deque(maxlen=buffer_size)

    def push(self, experience):
        self.buffer.append(experience)

    def sample(self, batch_size, return_all=False):
        # Sample a batch of experiences
        if return_all:
            sampled_experiences = self.buffer
        else:
            sampled_experiences = random.sample(self.buffer, batch_size)
        # Transpose the list of experiences, then convert each component to a NumPy array
        sampled_experiences = [np.array(x) for x in zip(*sampled_experiences)]
        # Ensure each component has at least 2 dimensions
        return [x if x.ndim >= 2 else np.expand_dims(x, axis=-1) for x in sampled_experiences]
    
    def compute_rewards_to_go(self, reward_idx, done_idx, gamma):
        returns = np.zeros(len(self.buffer))
        running_return = 0
        rewards = np.array([exp[reward_idx] for exp in self.buffer], dtype=np.float32)
        dones = np.array([exp[done_idx] for exp in self.buffer], dtype=np.float32)
        # Compute rewards-to-go in reverse
        for j in reversed(range(len(rewards))):
            if dones[j]:
                running_return = 0
            running_return = rewards[j] + gamma * running_return
            returns[j] = running_return
        # Append the computed returns to each experience
        for i in range(len(self.buffer)):
            self.buffer[i].append(returns[i])      # Append reward-to-go

    def __len__(self):
        return len(self.buffer)
    
    def __getitem__(self, idx):
        return self.buffer[idx]
    
    def clear(self):
        self.buffer.clear()

In [51]:
def compute_rewards_to_go(rewards, dones, gamma):
    returns = []
    running_return = 0
    rewards_flat = rewards.ravel()
    dones_flat = dones.ravel()
    # Iterate backwards over rewards and dones
    for reward, done in zip(reversed(rewards_flat), reversed(dones_flat)):
        if done:
            running_return = 0  # Reset at end of trajectory
        running_return = reward + gamma * running_return
        returns.insert(0, running_return)
    return torch.tensor(returns).reshape(rewards.shape)

In [52]:
memory = ReplayBuffer(int(500))

In [53]:
for i in range(10):
    memory.push([1*i,0,3*i,4*i])

In [54]:
memory[2]

[2, 0, 6, 8]

In [33]:
memory.buffer[0].append(1)

In [48]:
memory.buffer

deque([[0, 0, 0, 0],
       [1, 0, 3, 4],
       [2, 0, 6, 8],
       [3, 0, 9, 12],
       [4, 0, 12, 16],
       [5, 0, 15, 20],
       [6, 0, 18, 24],
       [7, 0, 21, 28],
       [8, 0, 24, 32],
       [9, 0, 27, 36]],
      maxlen=500)

In [49]:
memory.compute_rewards_to_go(0,1,0.99)
memory.buffer

deque([[0, 0, 0, 0, 42.23538240403106],
       [1, 0, 3, 4, 42.6620024283142],
       [2, 0, 6, 8, 42.08283073567091],
       [3, 0, 9, 12, 40.487707813809],
       [4, 0, 12, 16, 37.8663715291],
       [5, 0, 15, 20, 34.20845609],
       [6, 0, 18, 24, 29.503491],
       [7, 0, 21, 28, 23.7409],
       [8, 0, 24, 32, 16.91],
       [9, 0, 27, 36, 9.0]],
      maxlen=500)

In [23]:
e = random.sample(memory.buffer, 3)
e

[[0, 0, 0, 0], [2, 4, 6, 8], [8, 16, 24, 32]]

In [24]:
e2 = [np.array(x) for x in zip(*e)]
e2

[array([0, 2, 8]),
 array([ 0,  4, 16]),
 array([ 0,  6, 24]),
 array([ 0,  8, 32])]

In [22]:
[x if x.ndim >= 2 else np.expand_dims(x, axis=-1) for x in e2]

[array([[0],
        [3],
        [2]]),
 array([[0],
        [6],
        [4]]),
 array([[0],
        [9],
        [6]]),
 array([[ 0],
        [12],
        [ 8]])]